In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [35]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.inspection import permutation_importance

In [36]:
df = pd.read_csv("/content/banking_collections_cleaned.csv")
df

,Customer_ID,Name,Account_Number,Account_Type,Loan_Type,Loan_Amount,Outstanding_Amount,EMI_Amount,Due_Date,Payment_Status,...,Last_Payment_Date,Payment_Delay_Days,Region,Contact_Number,Email,Customer_Score,Risk_Level,Priority_Score,Est_Collection_Cost,Priority_Level
0,CUST00000,Richard Mccarty,81087F51-C9F,Current,Car Loan,299783,119495.19,2597.72,2023-12-05,Paid,...,2023-10-06,0,West,001-713-059-9538x4286,diane22@gmail.com,557,Medium,26.949519,700,Low
1,CUST00001,Patrick Kane,819326DB-44D,Current,Home Loan,46987,37830.59,1576.27,2023-04-07,Paid,...,2023-03-20,0,South,001-118-785-4275,edward97@yahoo.com,500,High,33.783059,900,Medium
2,CUST00002,Jean Morgan,A8682A8D-986,Current,Home Loan,259917,112988.74,4035.31,2024-10-31,Missed,...,No Payment Recorded,43,East,+1-158-183-9055,ifranklin@gmail.com,342,High,62.798874,1545,High
3,CUST00003,Michael Green,45516DAD-9C8,Credit,Personal Loan,198807,122047.34,3698.40,2024-01-23,Partially Paid,...,2023-11-10,84,South,828-258-5077,craig92@franklin-lindsey.com,806,Low,59.204734,1810,High
4,CUST00004,Tony Greene,18D78B45-41D,Savings,Home Loan,411507,113899.82,3254.28,2023-05-10,Partially Paid,...,2023-04-11,31,North,(611)277-3109x276,obaker@peterson.com,755,Low,31.889982,1015,Medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,CUST00995,Lucas Ingram,6717D08D-8BF,Savings,Car Loan,308726,98191.86,5455.10,2024-05-08,Missed,...,No Payment Recorded,73,North,+1-684-628-7728x808,ohodge@hotmail.com,386,High,76.319186,1995,High
996,CUST00996,Matthew Tran,30158651-FF0,Current,Car Loan,145881,42105.25,1203.01,2023-07-25,Paid,...,2023-05-26,0,South,147-363-0896x130,murphydominique@yahoo.com,387,High,34.210525,900,Medium
997,CUST00997,Robert Martinez,0B84F5D6-0E4,Current,Home Loan,293449,28638.37,520.70,2023-03-05,Partially Paid,...,2023-01-29,73,West,341.723.5709x3350,jjackson@yahoo.com,751,Low,44.363837,1645,Medium
998,CUST00998,Kimberly Ramirez,5E254A28-AF8,Savings,Personal Loan,492815,201018.27,6484.46,2023-09-29,Missed,...,No Payment Recorded,62,North,(226)320-5961x06171,katherineporter@gmail.com,755,Low,56.101827,1480,High


In [37]:
df.isnull().sum()

,0
Customer_ID,0
Name,0
Account_Number,0
Account_Type,0
Loan_Type,0
Loan_Amount,0
Outstanding_Amount,0
EMI_Amount,0
Due_Date,0
Payment_Status,0


In [38]:
df.columns

Index(['Customer_ID', 'Name', 'Account_Number', 'Account_Type', 'Loan_Type',
       'Loan_Amount', 'Outstanding_Amount', 'EMI_Amount', 'Due_Date',
       'Payment_Status', 'Collection_Agent', 'Last_Payment_Date',
       'Payment_Delay_Days', 'Region', 'Contact_Number', 'Email',
       'Customer_Score', 'Risk_Level', 'Priority_Score', 'Est_Collection_Cost',
       'Priority_Level'],
      dtype='object')

In [39]:
df.shape

(1000, 21)

In [40]:
feature_cols_numeric = [
    "Outstanding_Amount", "Payment_Delay_Days", "Customer_Score",
    "EMI_Amount", "Est_Collection_Cost"
]
feature_cols_categorical = ["Risk_Level", "Region", "Loan_Type", "Account_Type"]

df_model = df[feature_cols_numeric + feature_cols_categorical + ["Priority_Level"]].copy()

In [41]:
df_model

,Outstanding_Amount,Payment_Delay_Days,Customer_Score,EMI_Amount,Est_Collection_Cost,Risk_Level,Region,Loan_Type,Account_Type,Priority_Level
0,119495.19,0,557,2597.72,700,Medium,West,Car Loan,Current,Low
1,37830.59,0,500,1576.27,900,High,South,Home Loan,Current,Medium
2,112988.74,43,342,4035.31,1545,High,East,Home Loan,Current,High
3,122047.34,84,806,3698.40,1810,Low,South,Personal Loan,Credit,High
4,113899.82,31,755,3254.28,1015,Low,North,Home Loan,Savings,Medium
...,...,...,...,...,...,...,...,...,...,...
995,98191.86,73,386,5455.10,1995,High,North,Car Loan,Savings,High
996,42105.25,0,387,1203.01,900,High,South,Car Loan,Current,Medium
997,28638.37,73,751,520.70,1645,Low,West,Home Loan,Current,Medium
998,201018.27,62,755,6484.46,1480,Low,North,Personal Loan,Savings,High


In [42]:
encoders = {}
for col in feature_cols_categorical:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    encoders[col] = le

target_le = LabelEncoder()
y = target_le.fit_transform(df_model["Priority_Level"])
X = df_model[feature_cols_numeric + feature_cols_categorical]

In [43]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [44]:
model = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=3,
    random_state=42, class_weight="balanced"
)
model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=8, min_samples_leaf=3,
                       n_estimators=300, random_state=42)

In [45]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=target_le.classes_)
print(f"Accuracy: {acc:.3f}\n")
print(report)

Accuracy: 0.950

              precision    recall  f1-score   support

        High       0.97      0.98      0.98        66
         Low       0.91      0.96      0.94        54
      Medium       0.96      0.91      0.94        80

    accuracy                           0.95       200
   macro avg       0.95      0.95      0.95       200
weighted avg       0.95      0.95      0.95       200



In [46]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
perm = permutation_importance(model, X_test, y_test, n_repeats=20, random_state=42, n_jobs=-1)
perm_importances = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)

print("Gini importance:\n", importances, "\n")
print("Permutation importance:\n", perm_importances)

Gini importance:
 Est_Collection_Cost    0.338265
Payment_Delay_Days     0.214646
Outstanding_Amount     0.175103
Customer_Score         0.111129
EMI_Amount             0.090366
Risk_Level             0.055902
Region                 0.005387
Account_Type           0.005358
Loan_Type              0.003844
dtype: float64 

Permutation importance:
 Est_Collection_Cost    0.36675
Outstanding_Amount     0.20675
Payment_Delay_Days     0.14725
Customer_Score         0.08150
EMI_Amount             0.04775
Risk_Level             0.02800
Loan_Type              0.00450
Account_Type           0.00200
Region                 0.00000
dtype: float64
